**# Configuration**

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    col,
    lit
)

RAW_PATH = "/Volumes/workspace/default/nyc_hvfhv_data/raw/2026-01/"

BRONZE_TABLE = "workspace.default.bronze_hvfhv_trips"

SOURCE_MONTH = "2026-01"

In [0]:
raw_path = "/Volumes/workspace/default/nyc_hvfhv_data/raw/2026-01/"

display(dbutils.fs.ls(raw_path))

In [0]:
df_raw = spark.read.parquet(raw_path)
display(df_raw.limit(10))

In [0]:
df_raw.printSchema()

In [0]:
print(f"Number of columns: {len(df_raw.columns)}")

print("\nColumns:")
for column in df_raw.columns:
    print(column)

In [0]:
display(df_raw.limit(5))

In [0]:
row_count = df_raw.count()
print(f"Total Rows : {row_count}")

In [0]:
from pyspark.sql.functions import min, max

display(
    df_raw.select(
        min("pickup_datetime").alias("min_pickup_datetime"),
        max("pickup_datetime").alias("max-pickup_datetime")
    )
)


In [0]:
from pyspark.sql.functions import col, sum

null_counts = df_raw.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
])

display(null_counts)

**# Added Ingestion Metadata**

In [0]:
df_bronze = (
    df_raw
    .withColumn("_source_month", lit(SOURCE_MONTH))
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

In [0]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("_source_month")
    .saveAsTable(BRONZE_TABLE)
)

In [0]:
df_bronze_check = spark.read.table(BRONZE_TABLE)

print(f"Bronze Rows: {df_bronze_check.count():,}")
print(f"Bronze Columns: {len(df_bronze_check.columns)}")

display(df_bronze_check.limit(5))